# LOS/NLOS labels da coordinate CSV/JSON (+ DEM)

Per ogni punto mette un ricevitore a terra, calcola i satelliti da RINEX NAV e classifica **LOS/NLOS**:
- edifici → BVH ray-trace
- terreno → DEM horizon (opzionale ma **obbligatorio per Genova**)

Script: `experiments/label_los_nlos_coords.py`

### Accortezze Genova (da pipeline area-map / rqz)
1. **`--rx-alt-mode dem`**: non usare alt ellissoidale fissa 1.5 m (colline).
2. **`--dem-auto-download`** (o `--dem-path`): horizon terrain; senza DEM le colline non occludono.
3. **`--geoid-correction none`**: tile AWS elevation già ellissoidali.
4. Mesh OSM: passa lo stesso DEM a `fetch_osm_buildings_bbox.py --dem-path` così i footprint poggiano sul terreno.
5. Bbox città storica usata in passato: `S/W/N/E = 44.385 / 8.880 / 44.445 / 8.980`.

**Runtime:** GPU (T4) + build CUDA `_bvh`.  
**Repo:** `https://github.com/andrea444460/gnss_gpu_los_nlos` · branch `feature/cuda-preprocess-area-map`

In [ ]:
# 1) Verifica GPU
!nvidia-smi

In [ ]:
# 2) Clone / aggiorna repo
import os, shutil, subprocess

REPO = "/content/gnss_gpu"
BRANCH = "feature/cuda-preprocess-area-map"
REPO_URL = "https://github.com/andrea444460/gnss_gpu_los_nlos.git"
SCRIPT = f"{REPO}/experiments/label_los_nlos_coords.py"

def _has_script():
    return os.path.isfile(SCRIPT)

if os.path.isdir(REPO) and not _has_script():
    print("Cartella esiste ma manca lo script → rimuovo clone errato")
    shutil.rmtree(REPO)

if not os.path.isdir(REPO):
    !git clone -b {BRANCH} {REPO_URL} {REPO}
else:
    subprocess.run(["git", "-C", REPO, "fetch", REPO_URL, BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", REPO_URL, BRANCH], check=True)

os.chdir(REPO)
assert _has_script(), f"Manca {SCRIPT} — pusha lo script sul branch prima"
print("OK:", SCRIPT)
!git -C {REPO} rev-parse --short HEAD

In [ ]:
# 3) Dipendenze + build CUDA (T4 = sm_75)
import os
REPO = "/content/gnss_gpu"
os.chdir(REPO)

!pip install -q numpy matplotlib requests scipy rasterio pyproj
!apt-get -qq install -y cmake
!mkdir -p {REPO}/build
%cd {REPO}/build
!cmake .. -DCMAKE_CUDA_ARCHITECTURES=75
!make -j$(nproc)
%cd {REPO}

In [ ]:
# 4) Test import BVH
import os, sys
REPO = "/content/gnss_gpu"
os.chdir(REPO)
os.environ["PYTHONPATH"] = f"{REPO}/python:{REPO}/build"
sys.path[:0] = [f"{REPO}/python", f"{REPO}/build"]
import gnss_gpu._bvh
print("_bvh OK")

## 5) Dati

In `/content/los_work/data/`:
- `points.csv` / `points.json`
- NAV `*.rnx`
- mesh `triangles.npy` (o genera OSM sotto)

CSV: `id,lat,lon` (con `--rx-alt-mode dem` l’alt CSV viene ignorata).

In [ ]:
# 5a) Cartelle + esempio Genova centro
from pathlib import Path

WORK = Path("/content/los_work")
DATA = WORK / "data"
OUT = WORK / "results"
DATA.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

example = DATA / "points.csv"
if not example.exists():
    # due punti nel centro storico (bbox genova_zones.DEFAULT_CENTRO_BBOX)
    example.write_text(
        "id,lat,lon\n"
        "centro,44.4070,8.9330\n"
        "porto,44.4055,8.9280\n",
        encoding="utf-8",
    )
    print("Creato esempio Genova:", example)
else:
    print("Esiste già:", example)

print("File in data/:", sorted(p.name for p in DATA.iterdir()))

In [ ]:
# 5b) Upload (points / NAV / triangles opzionale)
from google.colab import files
from pathlib import Path
import shutil

DATA = Path("/content/los_work/data")
DATA.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    dest = DATA / name
    shutil.move(name, dest)
    print("→", dest)

In [ ]:
# 5c) Mesh OSM Genova + DEM (stesso DEM per footprint edifici)
import os, csv, math
from pathlib import Path

REPO = "/content/gnss_gpu"
DATA = Path("/content/los_work/data")
OUT = Path("/content/los_work/results")
os.environ["PYTHONPATH"] = f"{REPO}/python:{REPO}/build"

coords = DATA / "points.csv"
tri_out = OUT / "genova_osm_triangles.npy"
dem_out = OUT / "genova_area_dem.tif"
BUFFER_DEG = 0.02  # ~2 km intorno ai punti

lats, lons = [], []
with open(coords, newline="", encoding="utf-8") as fh:
    for row in csv.DictReader(fh):
        lats.append(float(row.get("lat") or row.get("lat_deg")))
        lons.append(float(row.get("lon") or row.get("lon_deg")))

south = min(lats) - BUFFER_DEG
north = max(lats) + BUFFER_DEG
west = min(lons) - BUFFER_DEG
east = max(lons) + BUFFER_DEG
# oppure bbox città completa usata in passato:
# south, west, north, east = 44.3850, 8.8800, 44.4450, 8.9800
print(f"OSM/DEM bbox: S={south:.5f} W={west:.5f} N={north:.5f} E={east:.5f}")

# DEM prima (serve anche ai footprint OSM)
from gnss_gpu.io.dem_download import download_dem_for_bbox
dem_path, ntiles = download_dem_for_bbox(
    south=south - 0.15, west=west - 0.15,  # padding ~15–20 km per horizon colline
    north=north + 0.15, east=east + 0.15,
    output_tif=dem_out, zoom=12,
)
print("DEM:", dem_path, "tiles=", ntiles)

!python {REPO}/experiments/fetch_osm_buildings_bbox.py \
  --south {south} --west {west} --north {north} --east {east} \
  --cache-json {OUT}/genova_osm_cache.json \
  --out-triangles {tri_out} \
  --dem-path {dem_out} \
  --geoid-correction none

print("Mesh:", tri_out, "exists=", tri_out.exists())

## 6) Label LOS/NLOS (Genova: DEM on)

`is_los` = edifici AND mask elev AND non bloccato da terreno  
`is_los_buildings` = solo edifici (ablation)

In [ ]:
# 6) Run labeling con DEM (consigliato Genova)
import os
from pathlib import Path

REPO = "/content/gnss_gpu"
DATA = Path("/content/los_work/data")
OUT = Path("/content/los_work/results")
os.chdir(REPO)
os.environ["PYTHONPATH"] = f"{REPO}/python:{REPO}/build"

COORDS = DATA / "points.csv"
NAV = next(DATA.glob("*MN.rnx"), None) or next(DATA.glob("*.rnx"), None)
TRIANGLES = OUT / "genova_osm_triangles.npy"
if not TRIANGLES.exists():
    TRIANGLES = next(DATA.glob("*triangles*.npy"), None)
DEM = OUT / "genova_area_dem.tif"

GPS_WEEK = 2320      # <-- imposta
GPS_TOW = 360000.0   # <-- imposta
ELEV_MASK = 10.0

assert COORDS.exists(), COORDS
assert NAV and Path(NAV).exists(), f"Manca NAV in {DATA}"
assert TRIANGLES and Path(TRIANGLES).exists(), "Manca mesh — lancia cella 5c"

OUT_CSV = OUT / "los_nlos_labels.csv"
cmd = f"""python experiments/label_los_nlos_coords.py \
  --coords {COORDS} \
  --nav {NAV} \
  --triangles {TRIANGLES} \
  --gps-week {GPS_WEEK} \
  --gps-tow {GPS_TOW} \
  --rx-alt-mode dem \
  --rx-ant-height-m 1.5 \
  --geoid-correction none \
  --elevation-mask-deg {ELEV_MASK} \
  --output-csv {OUT_CSV} \
  --summary-csv {OUT}/los_nlos_summary.csv"""

if DEM.exists():
    cmd += f" --dem-path {DEM}"
else:
    cmd += f" --dem-auto-download --dem-auto-out {OUT}/genova_area_dem.tif"

print(cmd)
get_ipython().system(cmd)

In [ ]:
# 7) Anteprima
import pandas as pd
from pathlib import Path

OUT = Path("/content/los_work/results")
labels = pd.read_csv(OUT / "los_nlos_labels.csv")
summary = pd.read_csv(OUT / "los_nlos_summary.csv")
display(summary)
display(labels.head(20))
print(labels.groupby(["point_id", "is_los"]).size())
if "terrain_blocked_visible" in labels.columns:
    print("terrain_blocked_visible per punto:")
    print(labels.groupby("point_id")["terrain_blocked_visible"].sum())

In [ ]:
# 8) Download
from google.colab import files
from pathlib import Path
OUT = Path("/content/los_work/results")
files.download(str(OUT / "los_nlos_labels.csv"))
files.download(str(OUT / "los_nlos_summary.csv"))